In [1]:
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
class GaussianMixtureModel:
    def __init__(self, n_categories: int, n_features: int):
        self.n_categories = n_categories
        
    def _initialize_kmeans(self, samples: np.ndarray):
        N, d = samples.shape # otteniamo dal dataset info sul numero di campioni e numero di features 
        k = self.n_categories # n_categories è il numero di clusters
        random_indices = np.random.choice(N, k, replace=False) # vengono selezionati i centrodi e con replace impostato a False si evita di ripescare lo stesso numero
        # assunzioni iniziali
        self._mu = samples[random_indices].copy() # i centrodi di partenza di ogni cluster vengono salvati in un nuovo array e diventano medie di ogni campana 
        self._pi = np.ones(k)/k # inizializzazione dei pesi assegnati ai centrodi: creazione di un array di k elementi identici (1/k), cosicché la somma sia sempre = 1
        self._sigma = np.ones((k, d)) # creazione di una matrice riempita con 1 (si assume che le campane siano perfettamente sferiche)
        # algoritmo k-means
        max_iter = 8
        for _ in range(max_iter): # ad ogni iterazione
            distance_matrix = np.zeros((N, k)) # inizialmente si crea una matrice riempita con 0 
            for m in range(k): # per ogni centroide m (per ogni colonna della matrice) (k = numero totale dei centroidi)
                distance_matrix[:, m] = np.sum((samples - self._mu[m])**2, axis=1) # ogni riga della colonna m viene riempita con il valore della distanza euclidea per ciascun campione del dataset 
            cluster_assignments = np.argmin(distance_matrix, axis=1) # array che contiene l'indice del cluster più vicino per ogni campione N 
            for m in range(k):
                points_in_cluster = samples[cluster_assignments == m] # sottoinsieme del dataset che contiene N assegnati al cluster m 
                if len(points_in_cluster) > 0:
                    # aggiornamento della posizione dei centroidi
                    self._mu[m] = np.mean(points_in_cluster, axis = 0) 
                    
    def log_likelihood(self, samples: np.ndarray) -> np.ndarray:
        """Computes logP(X|θ)"""
        N, d = samples.shape
        k = self.n_categories
        log_probs = np.zeros((N, k)) # matrice iniziale vuota usata per salvare le log-probabilities di ogni N per ogni cluster (campana)
        for m in range(k): # per ogni cluster 
            # calcolo della probabilità a priori che un qualsiasi N appartenga al cluster m
            log_pi = np.log(self._pi[m] + 1e-10)
            # calcolo della log-Gaussiana per una singola campana
            log_norm = -0.5 * (d*np.log(2*np.pi) + np.sum(np.log(self._sigma[m]))) # termine di normalizzazione della log-Gaussiana: assumendo covarianza diagonale,
                                                                                   # log|sigma| diventa la somma dei logaritmi delle varianze
            exponent = -0.5 * np.sum(((samples - self._mu[m])**2)/ self._sigma[m], axis=1) # esponente della log-Gaussiana: assumendo covarianza diagonale, 
                                                                                           # è possibile semplificare l'operazione matriciale nella somma delle distanze al quadrato divise per la varianza
            log_probs[:, m] = log_pi + log_norm + exponent # tutte le righe per ciascuna colonna m si riempiono con il risultato della somma
        return np.sum(np.logaddexp.reduce(log_probs, axis=1)) # ritorna la somma degli array in cui ogni valore corrisponde al risultato del log-sum-exp applicato ad una specifica riga

    def fit(self, samples: np.ndarray):
        """Fits the GMM parameters using the EM algorithm."""
        N, d = samples.shape
        k = self.n_categories
        self._initialize_kmeans(samples)
        max_iter = 100 # nonostante il controllo con epsilon, rimane comunque importante evitare di cadere in loop di calcolo infiniti
        epsilon = 1e-4 
        prev_log_likelihood = -np.inf # inizializzata per funzionare già alla prima iterazione
        for _ in range(max_iter):
            log_probs = np.zeros((N, k)) 
            # E-step
            for m in range(k):
                log_pi = np.log(self._pi[m] + 1e-10)
                log_norm = -0.5 * (d*np.log(2*np.pi) + np.sum(np.log(self._sigma[m])))
                exponent = -0.5 * np.sum(((samples - self._mu[m])**2)/ self._sigma[m], axis=1)
                log_probs[:, m] = log_pi + log_norm + exponent # numeratore nella formula delle responsabilities (nella formula della probabilità a posteriori)
            log_marginal = np.logaddexp.reduce(log_probs, axis=1, keepdims=True) # denominatore della formula delle responsabilities, keepdims=True mantiene le dimensioni per il broadcasting
            responsibilities = np.exp(log_probs-log_marginal) # calcolo della probabilità a posteriori (responsabilities)
                                                              # si converte la differenza dei logaritmi (rapporto tra numeratore e denominatore nella formula delle responsabilities) in probabilità tra 0 e 1
            current_log_likelihood = np.sum(log_marginal) # probabilità che tutti i campioni N siano stati generati dal GMM
            # M-step
            soft_counts = np.sum(responsibilities, axis=0) + 1e-10 # calcolo del numero probabilistico di punti assegnati al cluster m
            self._pi = soft_counts / N # calcolo aggiornamento pesi
            self._mu = (responsibilities.T @ samples) / soft_counts[:, np.newaxis] # aggiornamento della media calcolando il prodotto matriciale tra la trasposta di responsibilities e sample
                                                                                   # np.newaxis per il broadcasting
            # aggiornamento delle covarianze (diagonali)
            for m in range(k):
                diff = samples - self._mu[m] 
                weighted_sum_sq = np.sum(responsibilities[:, m][:, np.newaxis]*(diff**2), axis=0) # calcolo della distanza al quadrato dei punti dal nuovo centro
                self._sigma[m] = (weighted_sum_sq / soft_counts[m]) + 1e-10
            # l'algoritmo EM si ferma quando il miglioramento tra un'epoca e l'altra è minimo (o quando raggiunge il numero massimo di iterazioni)
            if abs(current_log_likelihood - prev_log_likelihood) < epsilon:
                break
            prev_log_likelihood = current_log_likelihood
            
    def bic(self, samples: np.ndarray) -> np.ndarray:
        """Computes the BIC score."""
        N, d = samples.shape
        k = self.n_categories
        total_log_likelihood = self.log_likelihood(samples) # calcola la likelihood totale sull'intero dataset
        num_parameters = (k-1)+(k*d)+(k*d) # calcolo del numero totale di parametri liberi considerando il numero delle dimensioni
        bic_score = total_log_likelihood - (num_parameters/2)*np.log(N)
        return bic_score

In [3]:
# load training set
train_df = pd.read_csv('train.csv')
print(f"train.csv loaded successfully. n={train_df.shape[0]}, d={train_df.shape[1]}")

# model selection using bayesian information score
seed_everything(9951)
candidate_categories = range(1, 7)
max_bic, best_k, best_gmm = -np.inf, None, None
for k in candidate_categories:
    gmm = GaussianMixtureModel(n_categories=k, n_features=train_df.shape[1])
    gmm.fit(train_df.values)
    ll = gmm.log_likelihood(train_df.values)
    bic_score = gmm.bic(train_df.values)
    print(f"k={k}\tBIC={bic_score:.4f}\tlogP(X|θ)={ll:.4f}")
    if bic_score > max_bic:
        max_bic = bic_score
        best_k = k
        best_gmm = gmm

# print training info
print()
print("Best model")
print(f"k:\t\t{best_k}")
print(f"BIC:\t\t{max_bic:.4f}")
print(f"logP(X|θ):\t{best_gmm.log_likelihood(train_df.values):.4f}")

train.csv loaded successfully. n=800, d=5
k=1	BIC=-8320.1771	logP(X|θ)=-8286.7541
k=2	BIC=-8052.0416	logP(X|θ)=-7981.8532
k=3	BIC=-7919.3844	logP(X|θ)=-7812.4306
k=4	BIC=-7869.8688	logP(X|θ)=-7726.1496
k=5	BIC=-7898.7302	logP(X|θ)=-7718.2457
k=6	BIC=-7927.0872	logP(X|θ)=-7709.8373

Best model
k:		4
BIC:		-7869.8688
logP(X|θ):	-7726.1496
